# Denoising and Variational Auto-encoders with PyTorch Lightning

The previous notebook trained auto-encoders to rebuild their input. This one first changes the task: recovering a clean image from a noisy version of it. Then it changes the model, to be able to generate new images. The data is Fashion-MNIST's: 60,000 training and 10,000 test images, 28 × 28 in grayscale like MNIST, but of 10 kinds of clothes and accessories.

1. Corrupting the images
2. A dense denoiser
3. An overcomplete code
4. A convolutional denoiser
5. A variational auto-encoder
6. Generating images
7. The effect of β

Each part reuses the data and models of the previous ones. No GPU needed: the notebook runs in 20 to 30 minutes on a CPU. A GPU (`Runtime > Change runtime type`) speeds it up.

## Installation and Imports

In [ ]:
!pip install -q lightning

In [ ]:
from collections import OrderedDict

import lightning
import matplotlib.pyplot as plt
import torch
import torchvision
from lightning.pytorch.callbacks import TQDMProgressBar
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.utilities.model_summary import ModelSummary
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

lightning.seed_everything(0)

## Loading Fashion-MNIST

In [ ]:
train_data = torchvision.datasets.FashionMNIST("data", train=True, download=True)
test_data = torchvision.datasets.FashionMNIST("data", train=False, download=True)
class_names = [
  "T-shirt",
  "Trouser",
  "Pullover",
  "Dress",
  "Coat",
  "Sandal",
  "Shirt",
  "Sneaker",
  "Bag",
  "Ankle boot",
]
input_dim = 28 * 28
# Pixels in [0, 1], like a sigmoid's output
X_train = train_data.data.reshape(-1, input_dim).float() / 255
y_train = train_data.targets
X_test = test_data.data.reshape(-1, input_dim).float() / 255
y_test = test_data.targets
print(X_train.shape, X_test.shape)


def show_images(*rows: torch.Tensor, titles: list | None = None) -> None:
  # Shows rows of 28 × 28 images, one row per array
  n = len(rows[0])
  _, axes = plt.subplots(
    len(rows), n, figsize=(n * 1.2, len(rows) * 1.3), squeeze=False
  )
  for row, images in zip(axes, rows):
    for ax, image in zip(row, images):
      ax.imshow(image.reshape(28, 28), cmap="gray_r")
      ax.axis("off")
  for ax, title in zip(axes[0], [] if titles is None else titles):
    ax.set_title(title, fontsize=8)
  plt.show()


show_images(X_train[:10], titles=[class_names[y] for y in y_train[:10]])

## 1. Corrupting the images

A denoiser learns to recover the clean image from a damaged version of it: its training examples are pairs (noisy image, clean image). Make the noisy images:

- `X_train_noisy` and `X_test_noisy`: `X_train` and `X_test`, plus a Gaussian noise of mean 0 and standard deviation 0.5 drawn independently for each pixel;
- then bring each pixel back into [0, 1], like those of a real image.

The cell after the solution shows noisy images above the clean ones.

**Useful documentation**

- [`torch.randn`](https://docs.pytorch.org/docs/stable/generated/torch.randn.html)
- [`torch.Generator`](https://docs.pytorch.org/docs/stable/generated/torch.Generator.html)
- [`torch.clamp`](https://docs.pytorch.org/docs/stable/generated/torch.clamp.html)

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `generator = torch.Generator().manual_seed(0)`, then `torch.randn(X_train.shape, generator=generator)`: a noise of mean 0 and standard deviation 1, of the images' shape, one draw per pixel; multiplied by 0.5, its standard deviation is 0.5
- `X.clamp(0, 1)`: the values below 0 become 0, those above 1 become 1
- check: `X_train_noisy.min()` is 0, `X_train_noisy.max()` is 1, and `X_train_noisy.shape` is `X_train`'s

A skeleton to copy into the exercise cell, replacing each `...`:

```python
generator = torch.Generator().manual_seed(0)
X_train_noisy = (X_train + 0.5 * torch.randn(...)).clamp(..., ...)
X_test_noisy = ...
```

### Solution

In [ ]:
generator = torch.Generator().manual_seed(0)
X_train_noisy = (X_train + 0.5 * torch.randn(X_train.shape, generator=generator)).clamp(
  0, 1
)
X_test_noisy = (X_test + 0.5 * torch.randn(X_test.shape, generator=generator)).clamp(
  0, 1
)

### The noisy images

In [ ]:
titles = [class_names[y] for y in y_test[:10]]
show_images(X_test_noisy[:10], X_test[:10], titles=titles)

## The Training Tools

They are given, and serve the whole notebook:

- `AutoEncoder`, a `LightningModule` that wraps a network and a loss: its input is the noisy image, its target the clean one; it initializes the network's weights like Keras, which keeps code ReLU units from dying;
- `make_loaders(X_noisy, X, X_test_noisy, X_test)`, which returns the training and validation `DataLoader`s of the pairs (noisy image, clean image), in batches of 256; the validation is done on the test images;
- `train(model, loaders, max_epochs, name)`, which trains an `AutoEncoder` and returns its last validation loss;
- `predict(network, X)`, which applies a network to all the images of `X`, batch by batch, without computing any gradient.

In [ ]:
class AutoEncoder(lightning.LightningModule):
  """Lightning wrapper: reconstruct the clean image from the noisy one."""

  def __init__(
    self,
    model: nn.Module,
    loss: nn.Module,
    learning_rate: float = 1e-3,
    input_shape: tuple[int, ...] = (input_dim,),
  ) -> None:
    super().__init__()
    self.save_hyperparameters(ignore=["model", "loss"])
    self.model = model
    self.loss = loss
    # An example input lets Lightning show the shape of each layer's
    # input and output tensors in the model's summary
    self.example_input_array = torch.zeros(1, *input_shape)
    # Glorot initialization, Keras's: with PyTorch's, most of the ReLU units
    # of a small code die (zero for every image)
    for layer in model.modules():
      if isinstance(layer, (nn.Linear, nn.Conv2d)):
        nn.init.xavier_uniform_(layer.weight)
        nn.init.zeros_(layer.bias)

  def forward(self, images: torch.Tensor) -> torch.Tensor:
    return self.model(images)

  def _step(self, batch: tuple[torch.Tensor, torch.Tensor], stage: str) -> torch.Tensor:
    noisy_images, images = batch
    loss = self.loss(self(noisy_images), images)
    self.log(f"{stage}_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
    return loss

  def training_step(
    self, batch: tuple[torch.Tensor, torch.Tensor], batch_index: int
  ) -> torch.Tensor:
    return self._step(batch, "train")

  def validation_step(
    self, batch: tuple[torch.Tensor, torch.Tensor], batch_index: int
  ) -> torch.Tensor:
    return self._step(batch, "val")

  def configure_optimizers(self) -> torch.optim.Optimizer:
    return torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)


def make_loaders(
  X_noisy: torch.Tensor,
  X: torch.Tensor,
  X_test_noisy: torch.Tensor,
  X_test: torch.Tensor,
) -> tuple[DataLoader, DataLoader]:
  """Training and validation loaders of the pairs (noisy image, clean image)."""
  train_loader = DataLoader(TensorDataset(X_noisy, X), batch_size=256, shuffle=True)
  val_loader = DataLoader(TensorDataset(X_test_noisy, X_test), batch_size=256)
  return train_loader, val_loader


def train(
  model: AutoEncoder,
  loaders: tuple[DataLoader, DataLoader],
  max_epochs: int,
  name: str,
) -> float:
  """Train model and return its last validation loss."""
  trainer = lightning.Trainer(
    max_epochs=max_epochs,
    accelerator="auto",
    devices=1,
    logger=CSVLogger("logs", name=name),
    # tqdm's progress bar, rather than rich's, which Lightning picks when rich
    # is installed (Colab): it froze some trainings
    callbacks=[TQDMProgressBar()],
    enable_checkpointing=False,
    enable_model_summary=False,
  )
  trainer.fit(model, *loaders)
  return trainer.callback_metrics["val_loss"].item()


@torch.no_grad()
def predict(model: nn.Module, X: torch.Tensor, batch_size: int = 256) -> torch.Tensor:
  """Apply a network to every image of X, batch by batch."""
  device = next(model.parameters()).device
  training = model.training
  model.eval()
  predictions = [model(batch.to(device)).cpu() for batch in X.split(batch_size)]
  # Give the network its mode back: fit no longer sets it to training mode
  model.train(training)
  return torch.cat(predictions)


loaders = make_loaders(X_train_noisy, X_train, X_test_noisy, X_test)

## 2. A dense denoiser

Build `autoencoder`, a dense denoiser with one hidden layer, and `model`, the `AutoEncoder` that wraps it:

- an `nn.Linear` layer from `input_dim` to `encoding_dim = 12` units, followed by a ReLU: the code;
- the output layer, from `encoding_dim` to `input_dim`, followed by an activation suited to pixels in [0, 1];
- the `nn.MSELoss()` loss.

Show its summary with `ModelSummary`. The cell after the solution trains it on the pairs (noisy image, clean image), then shows noisy test images, their denoised version and the clean image.

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- a single `nn.Sequential` is enough: `nn.Linear`, `nn.ReLU()`, `nn.Linear`, then the output activation
- the output: `nn.Sigmoid()`, which returns values in ]0, 1[, like the pixels
- `print(ModelSummary(model, max_depth=-1))`: 9.4 K parameters for the code's layer (784 × 12 + 12) and 10.2 K for the output (12 × 784 + 784), 19.6 K in all

A skeleton to copy into the exercise cell, replacing each `...`:

```python
encoding_dim = 12
autoencoder = nn.Sequential(nn.Linear(..., ...), nn.ReLU(), ..., ...)
model = AutoEncoder(autoencoder, loss=...)
print(ModelSummary(model, max_depth=-1))
```

### Solution

In [ ]:
encoding_dim = 12
autoencoder = nn.Sequential(
  nn.Linear(input_dim, encoding_dim),
  nn.ReLU(),
  nn.Linear(encoding_dim, input_dim),
  nn.Sigmoid(),  # output in ]0, 1[, like the pixels
)
model = AutoEncoder(autoencoder, loss=nn.MSELoss())
print(ModelSummary(model, max_depth=-1))

### Training

The input is the noisy image, the target the clean one. Top: the noisy images, then their denoised version, then the clean images.

In [ ]:
val_loss = train(model, loaders, max_epochs=20, name="dense_12")
print(f"Validation loss (MSE): {val_loss:.4f}")
X_test_denoised = predict(autoencoder, X_test_noisy)
show_images(X_test_noisy[:10], X_test_denoised[:10], X_test[:10], titles=titles)

## 3. An overcomplete code

A code larger than the input, overcomplete, leaves an ordinary auto-encoder room to copy its input, noise included (slide "Undercomplete or Overcomplete Code"). And a denoiser? Compare them, with a code of `encoding_dim = 1024` units, more than the 784 pixels:

- write `build_dense_autoencoder(encoding_dim)`, which returns a network shaped like part 2's (the code layer, with a ReLU, then the output with a sigmoid);
- `plain`, an ordinary auto-encoder, trained to rebuild the clean images: its `DataLoader`s are `make_loaders(X_train, X_train, X_test, X_test)`;
- `denoiser`, a network of the same shape, trained as a denoiser on `loaders`, the pairs (noisy image, clean image);
- 10 epochs each, with the loss `nn.MSELoss()`;
- apply both to `X_test_noisy`, measure the mean squared error of each against the clean images, and show the noisy images, `plain`'s output, `denoiser`'s, then the clean images.

Which one copies the noise? Why doesn't the other, although it has room to?

In [ ]:
def build_dense_autoencoder(encoding_dim: int) -> nn.Sequential: ...


# Your code here

### Additional hints

The calls and parameters to use:

- `build_dense_autoencoder` returns part 2's `nn.Sequential`, with `encoding_dim` as a parameter; `print(ModelSummary(AutoEncoder(plain, loss=nn.MSELoss()), max_depth=-1))` shows 1.6 M parameters (1,607,440)
- `plain_loaders = make_loaders(X_train, X_train, X_test, X_test)`: the input and the target are both the clean image
- `train(AutoEncoder(plain, loss=nn.MSELoss()), plain_loaders, max_epochs=10, name="plain_1024")` trains `plain` in place, and `train(..., loaders, ...)` the denoiser
- `((predict(network, X_test_noisy) - X_test) ** 2).mean()`: the error against the clean images

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def build_dense_autoencoder(encoding_dim: int) -> nn.Sequential:
  return nn.Sequential(nn.Linear(input_dim, encoding_dim), nn.ReLU(), ..., ...)


encoding_dim = 1024
plain, denoiser = build_dense_autoencoder(encoding_dim), ...
plain_loaders = make_loaders(...)
train(
  AutoEncoder(plain, loss=nn.MSELoss()), plain_loaders, max_epochs=10, name="plain_1024"
)
train(AutoEncoder(denoiser, loss=...), ..., max_epochs=..., name="denoiser_1024")

for name, network in [("Plain AE", plain), ("Denoiser", denoiser)]:
  error = ...
  print(f"{name}: MSE {error:.4f} against the clean images")
show_images(X_test_noisy[:10], ..., ..., X_test[:10], titles=titles)
```

### Solution

In [ ]:
def build_dense_autoencoder(encoding_dim: int) -> nn.Sequential:
  return nn.Sequential(
    nn.Linear(input_dim, encoding_dim),
    nn.ReLU(),
    nn.Linear(encoding_dim, input_dim),
    nn.Sigmoid(),
  )


encoding_dim = 1024  # more than the 784 pixels
plain, denoiser = (
  build_dense_autoencoder(encoding_dim),
  build_dense_autoencoder(encoding_dim),
)
# The plain AE: the input and the target are the clean image
plain_loaders = make_loaders(X_train, X_train, X_test, X_test)
train(
  AutoEncoder(plain, loss=nn.MSELoss()), plain_loaders, max_epochs=10, name="plain_1024"
)
# The denoiser: the noisy image as input, the clean image as target
train(
  AutoEncoder(denoiser, loss=nn.MSELoss()), loaders, max_epochs=10, name="denoiser_1024"
)

for name, network in [("Plain AE", plain), ("Denoiser", denoiser)]:
  error = ((predict(network, X_test_noisy) - X_test) ** 2).mean()
  print(f"{name}: MSE {error:.4f} against the clean images")
# At the top the noisy images, then the plain AE, the denoiser, the clean images
show_images(
  X_test_noisy[:10],
  predict(plain, X_test_noisy[:10]),
  predict(denoiser, X_test_noisy[:10]),
  X_test[:10],
  titles=titles,
)

The plain auto-encoder copies most of the noise: its code of 1,024 numbers leaves it room to learn a function close to the identity, and, having seen only clean images, nothing taught it to erase noise. Its error against the clean images, about 0.05, is more than three times the denoiser's, about 0.015. With the same overcomplete code, the denoiser returns clean clothes, sharper than with part 2's code of 12 (0.025): its target is the clean image, and each pixel of noise copied would increase its loss. That is the constraint of the slide "Undercomplete or Overcomplete Code": an overcomplete code is only useful if the task forbids copying, here through a noisy input.

## 4. A convolutional denoiser

A dense layer handles each pixel on its own; a convolution uses each pixel's neighborhood, which suits an image better. PyTorch's convolution layers expect 1 × 28 × 28 images, with an axis for the channels right after the batch's (a single channel in grayscale, three in color): the next cell puts the images back into that shape, and prepares the `DataLoader`s of these images.

In [ ]:
X_train_img = X_train.reshape(-1, 1, 28, 28)
X_test_img = X_test.reshape(-1, 1, 28, 28)
X_train_noisy_img = X_train_noisy.reshape(-1, 1, 28, 28)
X_test_noisy_img = X_test_noisy.reshape(-1, 1, 28, 28)
print(X_train_img.shape)
image_loaders = make_loaders(
  X_train_noisy_img, X_train_img, X_test_noisy_img, X_test_img
)

### The model

Build a fully convolutional `encoder` and `decoder`, then `autoencoder`, which chains them under these two names, and `model`, the `AutoEncoder` that wraps it. All the `nn.Conv2d` have 3 × 3 filters and `padding="same"`:

- `encoder`, on 1 × 28 × 28 images: an `nn.Conv2d` of 16 filters (ReLU) and a 2 × 2 `nn.MaxPool2d`, then an `nn.Conv2d` of 8 filters (ReLU) and a 2 × 2 `nn.MaxPool2d`. The code is the 8 × 7 × 7 tensor that comes out;
- `decoder`, the way back: an `nn.Conv2d` of 8 filters (ReLU) and an `nn.Upsample` with a factor of 2, an `nn.Conv2d` of 16 filters (ReLU) and an `nn.Upsample` with a factor of 2, then a last `nn.Conv2d` with 1 filter followed by a sigmoid, which returns a 1 × 28 × 28 image;
- `model`, with the dense denoisers' loss, `nn.MSELoss()`, and `input_shape=(1, 28, 28)`, an image's shape, for the summary.

Check the shapes with `ModelSummary`, and compare its number of parameters with part 3's dense denoiser. Why not end the decoder with an `nn.Upsample`?

**Useful documentation**

- [`torch.nn.Conv2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html)
- [`torch.nn.MaxPool2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html)
- [`torch.nn.Upsample`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Upsample.html)

In [ ]:
# Your code here

#### Additional hints

The calls and parameters to use:

- `nn.Conv2d(1, 16, 3, padding="same")`: 1 input channel, 16 filters of 3 × 3; `padding="same"` keeps the image's size; a small function `conv(in_channels, out_channels)` that returns it saves repeating these arguments
- `nn.MaxPool2d(2)` divides the height and the width by 2 (28 → 14 → 7), `nn.Upsample(scale_factor=2)` multiplies them by 2 by repeating each pixel (7 → 14 → 28)
- the last layer: `conv(16, 1)`, then `nn.Sigmoid()`; an `nn.Upsample` at the end of the network would return blocks of 2 × 2 identical pixels, and an output without a sigmoid could leave [0, 1]
- `print(ModelSummary(model, max_depth=2))`: the encoder's output is `[1, 8, 7, 7]`, the decoder's `[1, 1, 28, 28]` (the leading 1 is the example input's batch); 3.2 K parameters in all (3,217), against 1.6 M for part 3's dense denoiser: each 3 × 3 filter serves at every point of the image

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def conv(in_channels: int, out_channels: int) -> nn.Conv2d:
  return nn.Conv2d(in_channels, out_channels, 3, padding="same")


encoder = nn.Sequential(conv(1, 16), nn.ReLU(), nn.MaxPool2d(2), ..., ..., ...)
decoder = nn.Sequential(
  conv(8, 8), nn.ReLU(), nn.Upsample(scale_factor=2), ..., ..., ..., ..., ...
)
autoencoder = nn.Sequential(OrderedDict(encoder=encoder, decoder=decoder))
model = AutoEncoder(autoencoder, loss=..., input_shape=...)
print(ModelSummary(model, max_depth=2))
```

#### Solution

In [ ]:
def conv(in_channels: int, out_channels: int) -> nn.Conv2d:
  return nn.Conv2d(in_channels, out_channels, 3, padding="same")


encoder = nn.Sequential(
  conv(1, 16),
  nn.ReLU(),
  nn.MaxPool2d(2),  # 16 × 14 × 14
  conv(16, 8),
  nn.ReLU(),
  nn.MaxPool2d(2),  # 8 × 7 × 7, the code
)
decoder = nn.Sequential(
  conv(8, 8),
  nn.ReLU(),
  nn.Upsample(scale_factor=2),  # 8 × 14 × 14
  conv(8, 16),
  nn.ReLU(),
  nn.Upsample(scale_factor=2),  # 16 × 28 × 28
  # Smooths Upsample's blocks, then output in ]0, 1[
  conv(16, 1),
  nn.Sigmoid(),
)
autoencoder = nn.Sequential(OrderedDict(encoder=encoder, decoder=decoder))
model = AutoEncoder(autoencoder, loss=nn.MSELoss(), input_shape=(1, 28, 28))
print(ModelSummary(model, max_depth=2))

### Training

The same loss as the dense denoisers, the mean squared error, but only 5 epochs, against 20 for part 2's and 10 for part 3's: the comparison favors the dense ones.

In [ ]:
# 5 epochs, against 20 and 10 for the dense denoisers; the same loss, the MSE
val_loss = train(model, image_loaders, max_epochs=5, name="conv")
print(f"Validation loss (MSE): {val_loss:.4f}")
X_test_denoised = predict(autoencoder, X_test_noisy_img)
show_images(X_test_noisy[:10], X_test_denoised[:10], X_test[:10], titles=titles)

### On clean images

The denoiser has only seen noisy inputs. What does it return from a clean image?

In [ ]:
X_test_rebuilt = predict(autoencoder, X_test_img[:10])
show_images(X_test[:10], X_test_rebuilt, titles=titles)

On a clean image, the denoiser returns the garment's shape, but smoothed: it erases the fine details, like the pullover's logo, the shirt's checks or the sandal's straps, and forces the contrast. It learned to keep what looks like a garment and to erase the rest; to it, a fine detail looks like noise. Its mean squared error on the noisy images, about 0.021 after only 5 epochs, lies between those of the dense denoisers with a code of 12 (0.025) and of 1,024 (0.015), with 3,217 parameters instead of 19,612 and 1,607,440: each 3 × 3 filter serves at every point of the image, whereas each weight of a dense layer only links one pixel to one unit.

## 5. A variational auto-encoder

The previous notebook showed it: decoding a random code doesn't give a plausible image, as an auto-encoder's codes take up only a small part of the latent space, with no known shape. A variational auto-encoder (VAE) encodes each image into a distribution, $\mathcal{N}(\mu, \sigma^2)$, and a KL loss pushes these distributions towards $\mathcal{N}(0, I)$: one can then draw a code from $\mathcal{N}(0, I)$ and decode it.

The VAE's network is the one of the "A VAE in PyTorch" slide: it is given below, with its Lightning wrapper. It lacks the two pieces that make a VAE, which you write first.

### The sampling and the KL

Write, from the slides "Reparameterization Trick" and "Loss of a VAE":

- `Sampling`, an `nn.Module` whose `forward(z_mean, z_log_var)` takes $\mu$ and $\log \sigma^2$, and draws a code $z \sim \mathcal{N}(\mu, \sigma^2)$ with the reparameterization trick, so that the loss's gradient reaches $\mu$ and $\sigma$;
- `kl_divergence(z_mean, z_log_var)`, which returns, for each row of the batch, the KL between $\mathcal{N}(\mu, \sigma^2)$ and $\mathcal{N}(0, I)$: a tensor of one value per image.

The cell after the solution checks them on examples computable by hand.

**Useful documentation**

- [`torch.randn_like`](https://docs.pytorch.org/docs/stable/generated/torch.randn_like.html)
- [`torch.exp`](https://docs.pytorch.org/docs/stable/generated/torch.exp.html)
- [`torch.sum`](https://docs.pytorch.org/docs/stable/generated/torch.sum.html)

In [ ]:
class Sampling(nn.Module):
  def forward(self, z_mean: torch.Tensor, z_log_var: torch.Tensor) -> torch.Tensor:
    """A code drawn from N(z_mean, exp(z_log_var)), differentiable in both inputs."""
    ...


def kl_divergence(z_mean: torch.Tensor, z_log_var: torch.Tensor) -> torch.Tensor:
  """The KL between N(z_mean, exp(z_log_var)) and N(0, I), one value per row."""
  ...


# Your code here

#### Additional hints

The calls and parameters to use:

- the sampling: $z = \mu + \sigma \varepsilon$, with $\varepsilon \sim \mathcal{N}(0, I)$ drawn separately; `torch.randn_like(z_mean)` draws $\varepsilon$ with the shape of `z_mean`
- $\sigma = \exp\left(\frac{1}{2} \log \sigma^2\right)$: `torch.exp(0.5 * z_log_var)`
- the KL: $-\frac{1}{2} \sum_j \left(1 + \log \sigma_j^2 - \mu_j^2 - \sigma_j^2\right)$, the sum running over the code's dimensions, `dim=1`; $\sigma_j^2$ is `z_log_var.exp()`
- a check by hand: for $\mu = (0, 1)$ and $\sigma = (1, 1)$, that is a zero `z_log_var`, only the term $\mu_2^2 = 1$ counts, and the KL is 0.5

A skeleton to copy into the exercise cell, replacing each `...`:

```python
class Sampling(nn.Module):
  def forward(self, z_mean: torch.Tensor, z_log_var: torch.Tensor) -> torch.Tensor:
    eps = ...
    return z_mean + ... * eps


def kl_divergence(z_mean: torch.Tensor, z_log_var: torch.Tensor) -> torch.Tensor:
  return -0.5 * torch.sum(..., dim=1)
```

#### Solution

In [ ]:
class Sampling(nn.Module):
  def forward(self, z_mean: torch.Tensor, z_log_var: torch.Tensor) -> torch.Tensor:
    """A code drawn from N(z_mean, exp(z_log_var)), differentiable in both inputs."""
    eps = torch.randn_like(z_mean)  # ε ~ N(0, I), drawn separately
    return z_mean + torch.exp(0.5 * z_log_var) * eps  # z = μ + σ·ε


def kl_divergence(z_mean: torch.Tensor, z_log_var: torch.Tensor) -> torch.Tensor:
  """The KL between N(z_mean, exp(z_log_var)) and N(0, I), one value per row."""
  return -0.5 * torch.sum(1 + z_log_var - z_mean**2 - z_log_var.exp(), dim=1)

### Check

The KL of $\mathcal{N}((0, 1), I)$ is 0.5; drawn from $\mathcal{N}(0, 4)$, that is $\log \sigma^2 = \log 4$, codes have a mean close to 0 and a standard deviation close to 2 on each dimension.

In [ ]:
print("KL:", kl_divergence(torch.tensor([[0.0, 1.0]]), torch.zeros(1, 2)))
z = Sampling()(torch.zeros(10_000, 2), torch.full((10_000, 2), 4.0).log())
print("Mean:", z.mean(dim=0), "standard deviation:", z.std(dim=0))

### The model

They are given:

- `VAE`, an `AutoEncoder` for a network that returns the reconstruction, `z_mean` and `z_log_var`: its loss is the reconstruction's, plus `beta` times the mean of `kl_divergence` over the batch (slide "Sampling and Loss in PyTorch");
- `VariationalAutoEncoder`, the network of the "A VAE in PyTorch" slide, which draws its code with `Sampling`;
- `network`, a `VariationalAutoEncoder` with `latent_dim = 2` dimensions, `model`, the `VAE` that wraps it, with the `nn.BCELoss()` loss, and `vae_loaders`, its `DataLoader`s: the input and the target are both the clean image.

The summary shows 404 K parameters in all (404,244), almost all in the encoder's hidden layer and in the decoder's output.

In [ ]:
class VAE(AutoEncoder):
  """Lightning wrapper of a VAE: the reconstruction loss plus beta times the KL."""

  def __init__(
    self,
    model: nn.Module,
    loss: nn.Module,
    beta: float = 1 / 784,
    learning_rate: float = 1e-3,
  ) -> None:
    super().__init__(model, loss, learning_rate)
    self.beta = beta  # the KL's weight

  def _step(self, batch: tuple[torch.Tensor, torch.Tensor], stage: str) -> torch.Tensor:
    inputs, images = batch
    reconstructions, z_mean, z_log_var = self(inputs)
    kl = kl_divergence(z_mean, z_log_var)
    loss = self.loss(reconstructions, images) + self.beta * kl.mean()
    self.log(f"{stage}_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
    return loss


class VariationalAutoEncoder(nn.Module):
  """The VAE's network: an encoder with two outputs, the sampling, a decoder."""

  def __init__(self, latent_dim: int) -> None:
    super().__init__()
    self.hidden = nn.Sequential(nn.Linear(input_dim, 256), nn.ReLU())
    self.z_mean = nn.Linear(256, latent_dim)
    self.z_log_var = nn.Linear(256, latent_dim)  # log σ², any real number
    self.sampling = Sampling()
    self.decoder = nn.Sequential(
      nn.Linear(latent_dim, 256),
      nn.ReLU(),
      nn.Linear(256, input_dim),
      nn.Sigmoid(),
    )

  def encode(self, images: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """z_mean and z_log_var of a batch of images."""
    h = self.hidden(images)
    return self.z_mean(h), self.z_log_var(h)

  def forward(
    self, images: torch.Tensor
  ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """The reconstructions, z_mean and z_log_var of a batch of images."""
    z_mean, z_log_var = self.encode(images)
    z = self.sampling(z_mean, z_log_var)  # z = μ + σ·ε
    return self.decoder(z), z_mean, z_log_var


# beta = 1 / 784: the BCELoss loss is a mean over the 784 pixels,
# whereas the KL is a sum over the code's dimensions
vae_loaders = make_loaders(X_train, X_train, X_test, X_test)
latent_dim = 2
network = VariationalAutoEncoder(latent_dim)
model = VAE(network, loss=nn.BCELoss())  # + KL
print(ModelSummary(model, max_depth=2))

### Training and latent space

The target is the clean image itself. Each point of the cloud is a test image's $\mu$, colored by its class, which `show_latent` draws, a function that serves again in part 7.

In [ ]:
def show_latent(network: VariationalAutoEncoder, ax: plt.Axes) -> None:
  # The cloud of the test images' μ, colored by their class
  with torch.no_grad():
    test_means, _ = network.encode(X_test)
  for label, name in enumerate(class_names):
    points = test_means[y_test == label]
    ax.scatter(points[:, 0], points[:, 1], s=2, alpha=0.5, label=name)
  ax.set_xlabel("$z_1$")
  ax.set_ylabel("$z_2$")


train(model, vae_loaders, max_epochs=20, name="vae")
_, ax = plt.subplots(figsize=(7, 6))
show_latent(network, ax)
ax.legend(markerscale=5, fontsize=8)
plt.show()

Almost all the test images' μ stay between -3 and 3 on each axis, around the origin: the KL pulls each distribution towards $\mathcal{N}(0, I)$. Each class takes up its own region. The shoes (sandals, sneakers, ankle boots) gather on one side of the cloud, cut off from the rest by an almost empty band; the bags have their region, and so do the trousers, next to the dresses and T-shirts. The other tops (pullovers, coats, shirts) overlap: they look alike, and two dimensions aren't enough to separate them.

## 6. Generating images

As at the end of the "A VAE in PyTorch" slide, the next cell draws 16 codes from $\mathcal{N}(0, I)$ and decodes them with `network.decoder`. Do the images look like clothes? Compare them with the images decoded from noise in the previous notebook.

In [ ]:
# z ~ N(0, I)
z_new = torch.randn(16, latent_dim, generator=torch.Generator().manual_seed(0))
generated = predict(network.decoder, z_new)
show_images(generated)

Most of them look like clothes: tops, trousers, shoes, blurry like every image of a two-dimensional VAE. A few faint ones mix two kinds of clothes: their code fell between two regions, where the decoder hesitates. In the previous notebook, a random code gave only blots: here, the KL put the training images' codes where $\mathcal{N}(0, I)$ draws its own, and the decoder knows how to decode them. A larger latent space would give sharper images, but one could no longer draw its map.

### The latent space's map

With 2 dimensions, one can decode a regular grid of codes, from -2 to 2 on each axis, where $\mathcal{N}(0, I)$ draws almost all its points: each cell shows what the decoder returns at that point. Compare with part 5's cloud.

In [ ]:
n = 12
grid = torch.linspace(-2, 2, n).tolist()
# z_2 decreases from top to bottom, like the cloud's vertical axis
codes = torch.tensor([[z1, z2] for z2 in grid[::-1] for z1 in grid])
images = predict(network.decoder, codes).reshape(n, n, 28, 28)
plt.figure(figsize=(8, 8))
plt.imshow(images.permute(0, 2, 1, 3).reshape(n * 28, n * 28), cmap="gray_r")
plt.axis("off")
plt.show()

## 7. The effect of β

The slide "Loss of a VAE" says it: without the KL, we are back to an auto-encoder; with a larger $\beta$, the latent space is more regular, but the images blurrier. Check it:

- train two more `VariationalAutoEncoder`s with `latent_dim` dimensions, each wrapped in a `VAE`, for 20 epochs like part 5's: one with `beta=0`, the other with a $\beta$ ten times larger than part 5's, `10 / 784`;
- plot side by side, with `show_latent`, the $\mu$ clouds of the three VAEs ($\beta$ = 0, 1/784 and 10/784);
- decode, with each, part 6's 16 codes `z_new`, and show the three rows of images.

What become of the cloud and the images without the KL? With a $\beta$ ten times larger?

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `VAE(beta_network, loss=nn.BCELoss(), beta=0.0)`, then `train(..., vae_loaders, max_epochs=20, name="vae_beta_0")`: `beta_network` is trained in place
- a dictionary `{"0": ..., "1/784": network, "10/784": ...}` of the three networks keeps part 5's VAE and gives the figures' titles
- `_, axes = plt.subplots(1, 3, figsize=(18, 5))`, then, for each network, `show_latent(beta_network, ax)` and `ax.set_title(...)`; each cloud has its own axes: read their ticks
- `predict(beta_network.decoder, z_new)`: 16 images, which `show_images` shows
- the two trainings take a few minutes

A skeleton to copy into the exercise cell, replacing each `...`:

```python
networks = {
  "0": VariationalAutoEncoder(latent_dim),
  "1/784": network,
  "10/784": ...,
}
for beta_name, beta in [("0", 0.0), ("10/784", ...)]:
  train(
    VAE(networks[beta_name], loss=..., beta=...),
    vae_loaders,
    max_epochs=...,
    name=f"vae_beta_{beta:.4f}",
  )

_, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (beta_name, beta_network) in zip(axes, networks.items()):
  ...
  ax.set_title(f"β = {beta_name}")
plt.show()

for beta_name, beta_network in networks.items():
  print(f"β = {beta_name}")
  show_images(...)
```

### Solution

In [ ]:
networks = {
  "0": VariationalAutoEncoder(latent_dim),
  "1/784": network,  # part 5's VAE
  "10/784": VariationalAutoEncoder(latent_dim),
}
for beta_name, beta in [("0", 0.0), ("10/784", 10 / 784)]:
  train(
    VAE(networks[beta_name], loss=nn.BCELoss(), beta=beta),
    vae_loaders,
    max_epochs=20,
    name=f"vae_beta_{beta:.4f}",
  )

_, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (beta_name, beta_network) in zip(axes, networks.items()):
  show_latent(beta_network, ax)
  ax.set_title(f"β = {beta_name}")
axes[0].legend(markerscale=5, fontsize=8)
plt.show()

for beta_name, beta_network in networks.items():
  print(f"β = {beta_name}")
  show_images(predict(beta_network.decoder, z_new))

Without the KL ($\beta = 0$), the VAE becomes an auto-encoder again: nothing holds the $\mu$ back, they spread over tens of units, and each class goes off in its own direction, far from the origin. The 16 codes drawn from $\mathcal{N}(0, I)$ then all fall into a small central area where hardly any training image has its code: the decoder makes only blots of them. With a $\beta$ ten times larger (the slide's $\beta = 10$, as the reconstruction loss is here a mean over the 784 pixels), the cloud is tighter, between about -2 and 2, and the classes touch more; the generated images are all clothes, without the mixes of two kinds that $\beta = 1/784$ sometimes gives, but smoother, with no detail. The KL's weight sets that trade-off: a latent space where any code drawn can be decoded, against faithful reconstructions.